# Self-RAG
### Metacognitive retrieval — simulating the four reflection tokens with prompted `gpt-4o-mini`

Real Self-RAG fine-tunes a model to emit special reflection tokens (`RETRIEVE`, `RELEVANCE`, `SUPPORT`, `UTILITY`) as part of its own generation. Fine-tuning is out of scope for a notebook, so each checkpoint below is simulated as one small, structured prompt to `gpt-4o-mini` — the mechanism is identical, only *where* the intelligence lives differs (external prompt vs. internal weights).

Corpus: `OWASP Top 10 for LLMs (2025)`.

## Step 1: Build the pipeline

In [1]:
!pip install langchain langchain-community langchain-openai langchain-text-splitters faiss-cpu pypdf python-dotenv -q


[notice] A new release of pip is available: 25.0.1 -> 26.2.1
[notice] To update, run: python.exe -m pip install --upgrade pip


In [2]:
from dotenv import load_dotenv
load_dotenv()

from langchain_community.document_loaders import PyPDFLoader
from langchain_text_splitters import RecursiveCharacterTextSplitter
from langchain_community.vectorstores import FAISS
from langchain_openai import OpenAIEmbeddings, ChatOpenAI

pages = PyPDFLoader("OWASP-Top-10-for-LLMs-v2025.pdf").load()
chunks = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200).split_documents(pages)

embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
vector_store = FAISS.from_documents(chunks, embeddings)
llm = ChatOpenAI(model="gpt-4o-mini", temperature=0)

print(f"Loaded {len(pages)} pages -> {len(chunks)} chunks -> {vector_store.index.ntotal} vectors")

C:\Users\shiva\AppData\Local\Temp\ipykernel_26532\322707662.py:4: DeprecationWarning: `langchain-community` is being sunset and is no longer actively maintained. See https://github.com/langchain-ai/langchain-community/issues/674 for details and migration guidance toward standalone integration packages.
  from langchain_community.document_loaders import PyPDFLoader


C:\Users\shiva\.pyenv\pyenv-win\versions\3.12.10\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Error -3 while decompressing data: invalid code lengths set


Loaded 45 pages -> 131 chunks -> 131 vectors


## Step 2: RETRIEVE token — is retrieval even needed?
Skips retrieval entirely for queries that don't need the document at all.

In [3]:
def token_retrieve(query):
    prompt = f"""Does answering the message below require looking up the OWASP LLM Top 10 document,
or can it be answered directly (e.g. it's conversational, or general knowledge)?
Answer with exactly one word: RETRIEVE or DIRECT.

Message: {query}
Answer:"""
    label = llm.invoke(prompt).content.strip().upper()
    return "RETRIEVE" if "RETRIEVE" in label else "DIRECT"

## Step 3: RELEVANCE token — grade each retrieved chunk
Scores every chunk independently and discards the ones that don't actually address the query, before they ever reach generation.

In [4]:
def token_relevance(query, chunk):
    prompt = f"""Is the passage below relevant to answering the question? Answer with exactly one word: RELEVANT or IRRELEVANT.

Question: {query}
Passage: {chunk.page_content}
Answer:"""
    label = llm.invoke(prompt).content.strip().upper()
    return "IRRELEVANT" not in label and "RELEVANT" in label  # "IRRELEVANT" contains "RELEVANT" as a substring, so check it first

def filter_relevant(query, docs):
    return [d for d in docs if token_relevance(query, d)]

A quick sanity check with a genuinely relevant chunk and a genuinely unrelated one, so the grader's behavior is verified before it's trusted inside the pipeline.

In [5]:
from langchain_core.documents import Document

sample_query = "What is Prompt Injection?"
on_topic = vector_store.similarity_search(sample_query, k=1)[0]
off_topic = Document(page_content="The recipe for chocolate chip cookies requires butter, sugar, flour, and eggs baked at 350F for 12 minutes.")

print("on-topic  ->", token_relevance(sample_query, on_topic))
print("off-topic ->", token_relevance(sample_query, off_topic))

on-topic  -> True


off-topic -> False


## Step 4: Generate, then SUPPORT token — is every claim grounded?
If the generated answer contains claims the filtered context doesn't support, regenerate with a stricter grounding instruction instead of returning it as-is.

In [6]:
GEN_PROMPT = """Answer the question using only the following context.

Context:
{context}

Question: {query}
Answer:"""

STRICT_GEN_PROMPT = """Answer the question using ONLY facts explicitly stated in the context below.
If the context does not fully support a claim, omit it rather than guess.

Context:
{context}

Question: {query}
Answer:"""

def generate(query, docs, strict=False):
    context = "\n\n".join(d.page_content for d in docs)
    prompt = (STRICT_GEN_PROMPT if strict else GEN_PROMPT).format(context=context, query=query)
    return llm.invoke(prompt).content.strip()

def token_support(answer, docs):
    context = "\n\n".join(d.page_content for d in docs)
    prompt = f"""Is every claim in the answer below directly supported by the context? Answer with exactly one word: SUPPORTED or UNSUPPORTED.

Context:
{context}

Answer: {answer}
Verdict:"""
    label = llm.invoke(prompt).content.strip().upper()
    return "UNSUPPORTED" in label

## Step 5: UTILITY token — is the final answer actually useful?
If the answer is judged incomplete or unhelpful, re-retrieve with a wider net and try once more.

In [7]:
def token_utility(query, answer):
    prompt = f"""Does the answer below fully and usefully answer the question? Answer with exactly one word: USEFUL or LOW_UTILITY.

Question: {query}
Answer: {answer}
Verdict:"""
    label = llm.invoke(prompt).content.strip().upper()
    return "LOW_UTILITY" in label

## Step 6: Wire the four checkpoints into one pipeline

In [8]:
def self_rag(query, k=6):
    trace = {"query": query}

    if token_retrieve(query) == "DIRECT":
        trace["retrieve"] = "DIRECT (skipped)"
        trace["answer"] = llm.invoke(query).content.strip()
        return trace
    trace["retrieve"] = "RETRIEVE"

    docs = vector_store.similarity_search(query, k=k)
    relevant = filter_relevant(query, docs)
    trace["relevance"] = f"{len(relevant)}/{len(docs)} chunks kept"
    if not relevant:
        relevant = docs[:1]  # avoid an empty-context edge case

    answer = generate(query, relevant)
    if token_support(answer, relevant):
        trace["support"] = "UNSUPPORTED -> regenerated with strict grounding"
        answer = generate(query, relevant, strict=True)
    else:
        trace["support"] = "SUPPORTED"

    if token_utility(query, answer):
        trace["utility"] = "LOW_UTILITY -> re-retrieved with wider k"
        docs2 = vector_store.similarity_search(query, k=k + 6)
        relevant2 = filter_relevant(query, docs2) or docs2
        answer = generate(query, relevant2)
    else:
        trace["utility"] = "USEFUL"

    trace["answer"] = answer
    return trace

## Step 7: Run it — a conversational message, a direct lookup, and a broad question
The broad question is the same style of query that plain RAG struggles with: it's vague enough that a chunk-relevance filter should visibly discard some of what similarity search returns.

In [9]:
for q in [
    "Thanks, that explanation was really helpful!",
    "According to LLM07, what does System Prompt Leakage warn about?",
    "According to the OWASP Top 10 for LLMs, what are the main ways an LLM application can be attacked?",
]:
    trace = self_rag(q)
    print(f"Query: {trace['query']}")
    for step in ("retrieve", "relevance", "support", "utility"):
        if step in trace:
            print(f"  {step.upper():10s}: {trace[step]}")
    print(f"  ANSWER    : {trace['answer']}\n")

Query: Thanks, that explanation was really helpful!
  RETRIEVE  : DIRECT (skipped)
  ANSWER    : You're welcome! I'm glad to hear that you found the explanation helpful. If you have any more questions or need further clarification on anything, feel free to ask!



Query: According to LLM07, what does System Prompt Leakage warn about?
  RETRIEVE  : RETRIEVE
  RELEVANCE : 5/6 chunks kept
  SUPPORT   : SUPPORTED
  UTILITY   : USEFUL
  ANSWER    : LLM07 warns about the risk of system prompt leakage in LLMs, which refers to the potential exposure of sensitive information contained within system prompts or instructions that guide the model's behavior. This leakage can facilitate other attacks, such as prompt injection or unauthorized access, especially if the prompts include sensitive data like credentials or connection strings. The document emphasizes that system prompts should not be considered secrets and that sensitive data should not be included in them, as the real security risk lies in the improper handling of sensitive information and the potential for bypassing security controls.



Query: According to the OWASP Top 10 for LLMs, what are the main ways an LLM application can be attacked?
  RETRIEVE  : RETRIEVE
  RELEVANCE : 6/6 chunks kept
  SUPPORT   : SUPPORTED
  UTILITY   : USEFUL
  ANSWER    : According to the OWASP Top 10 for LLM Applications, the main ways an LLM application can be attacked include:

1. **XSS Attacks**: An attacker can manipulate the LLM to include malicious JavaScript in outputs, such as email content, leading to cross-site scripting (XSS) attacks if the output is not properly sanitized.

2. **Sensitive Information Disclosure**: LLMs can inadvertently expose sensitive information, such as personal identifiable information (PII) or confidential business data, through their outputs.

3. **Insecure Code Generation**: When LLMs generate code, they may introduce vulnerabilities like SQL injection or insecure data handling methods, especially if the generated code is not thoroughly reviewed.

4. **Hallucinated Packages**: Attackers can exploit the

## Try it yourself
1. Ask a question the OWASP document doesn't cover at all and see whether RELEVANCE discards every chunk, and how the empty-context fallback behaves.
2. Force an UNSUPPORTED verdict by asking the generator (via a modified `GEN_PROMPT`) to add one invented statistic, and watch the SUPPORT check catch it.
3. Compare token cost: Self-RAG makes up to 2 + 2·k + 2 extra LLM calls per query versus plain RAG's 1 — measure that overhead directly.